# Scenario 11: Pure OpenTelemetry Integration with Galileo (Customer Pattern)

This notebook shows how to send AI traces to Galileo using **only standard OpenTelemetry and OpenInference** — no `galileo.otel` helpers, no Galileo schema classes. It mirrors the pattern many enterprise teams already have in production: a generic OTel pipeline where Galileo is just another OTLP backend.

The only Galileo-specific parts are:

1. The OTLP endpoint: `{api_url}/otel/traces`
2. Three HTTP headers: `Galileo-API-Key`, `project`, `logstream`

That's it. If you point the exporter at a different backend tomorrow, the rest of your instrumentation code does not change.

By the end, you'll understand how to:

1. Build a **standard `TracerProvider` + `BatchSpanProcessor` + `OTLPSpanExporter`** aimed at Galileo
2. Auto-instrument OpenAI via the OpenInference `OpenAIInstrumentor`
3. Create **workflow, retriever, and tool spans by hand** using `tracer.start_as_current_span` and OpenInference semantic-convention attributes
4. Attach business-specific attributes like `workflow.name`, `usecase.id`, `step.name`, `http.url` (as in the customer's sample)
5. Wrap the span boilerplate in **reusable decorators**
6. Flush and shut down the pipeline cleanly

> **Production note:** In a web service, you'd also add `FastAPIInstrumentor.instrument_app(app)` and `RequestsInstrumentor().instrument()` (as in the customer's `init_telemetry()`) to get HTTP spans alongside AI spans. In a notebook there's no server to instrument, so we focus on the AI tracing path.

## How this differs from Scenario 10

| Aspect | Scenario 10 | Scenario 11 (this notebook) |
|---|---|---|
| Span processor | `galileo.otel.GalileoSpanProcessor` | Raw `BatchSpanProcessor` + `OTLPSpanExporter` |
| Custom spans | `WorkflowSpan`/`ToolSpan`/`RetrieverSpan` + `start_galileo_span` | `tracer.start_as_current_span` + OpenInference `SpanAttributes` |
| Coupling to Galileo SDK | Moderate (span helpers) | Minimal (endpoint + headers only) |
| Migration cost to a different OTLP backend | Refactor | Change one URL |

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`


## Step 0: Load Environment Variables

Same env-loading pattern used across every scenario in this repo — find a `.env` in the repo root or current directory, load it, and stop at the first match.

In [11]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Imports and Constants

Notice what is **not** here: no `from galileo import otel`, no `GalileoSpanProcessor`, no `start_galileo_span`, no `WorkflowSpan`/`ToolSpan`/`RetrieverSpan`. The only Galileo imports are:

- `galileo_context` — to create the project/log stream so there is somewhere for traces to land
- `GalileoPythonConfig` — to look up the API URL and API key that point the raw OTel exporter at Galileo
- `delete_project` — for the cleanup cell at the end

Everything else is vanilla OpenTelemetry and OpenInference.

### How Galileo extracts fields from OTel spans

Galileo's OTLP backend automatically maps span attributes with a fallback chain (attributes first, then span events). No per-project configuration is needed.

| Field | Attribute (primary) | Fallback |
|---|---|---|
| Input/output messages | `gen_ai.input.messages` / `gen_ai.output.messages` | Span events (`gen_ai.user.message`, `gen_ai.assistant.message`) |
| System instructions | `gen_ai.system_instructions` | First system message in `gen_ai.input.messages` |
| Token metrics | `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` | OpenInference `llm.token_count.prompt` / `.completion` |
| Context/documents | Retriever span output (list of documents) | Extracted automatically from retriever spans |
| Ground truth | `galileo.dataset.input` / `galileo.dataset.output` | Resource attributes for evaluation metrics |

### How Galileo identifies span types

| Span type | Primary discriminator | Other required attributes |
|---|---|---|
| Retriever | `db.operation` = `"search"` or `"query"` | `gen_ai.input.messages`, `gen_ai.output.messages` (document list) |
| Tool | `gen_ai.operation.name` = `"execute_tool"` | `gen_ai.tool.name`, `gen_ai.tool.call.arguments` / `.result` |
| Workflow | (parent span wrapping children) | `gen_ai.input.messages`, `gen_ai.output.messages` |
| LLM | Auto-instrumented by OpenInference | `gen_ai.operation.name` = `"chat"`, model, messages |

All custom spans should set `gen_ai.system` to identify the span source.


In [12]:
import functools
import json
import os
from urllib.parse import urljoin

import openai
from galileo import galileo_context
from galileo.config import GalileoPythonConfig
from galileo.projects import delete_project
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry import trace
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor

PROJECT_NAME = 'GalileoEval_S11_PureOTel'
LOG_STREAM_NAME = 'pure-otel'
SERVICE_NAME = 'galileo-pure-otel-demo'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, SERVICE_NAME, MODEL

('GalileoEval_S11_PureOTel',
 'pure-otel',
 'galileo-pure-otel-demo',
 'gpt-4o-mini')

## Step 1: Initialize Galileo (project + log stream only)

`galileo_context.init()` creates the project/log stream and populates `GalileoPythonConfig` with the resolved `api_url` and `api_key` for the current environment. We read those values back so we can build the raw OTLP exporter against them.

This is the **only** time the Galileo SDK is involved in the tracing path — after this step, everything is standard OTel.

In [13]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

GALILEO_API_URL = str(config.api_url)
GALILEO_API_KEY = config.api_key.get_secret_value() if config.api_key else None

# Customers who send OTLP through an internal collector (as in the sample
# script) can override the endpoint here via an env var and leave everything
# else untouched. This is the single line that points the pipeline at Galileo.
OTEL_TRACES_ENDPOINT = os.environ.get(
    'OTEL_EXPORTER_OTLP_TRACES_ENDPOINT',
    urljoin(GALILEO_API_URL if GALILEO_API_URL.endswith('/') else GALILEO_API_URL + '/', 'otel/traces'),
)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    log_stream_url = 'Galileo context initialized, but no log stream URL was returned.'
    print(log_stream_url)

print(f'\nOTLP endpoint -> {OTEL_TRACES_ENDPOINT}')

https://console.demo-v2.galileocloud.io/project/32921366-fb69-4449-99d9-05453ebaebd9
https://console.demo-v2.galileocloud.io/project/32921366-fb69-4449-99d9-05453ebaebd9/log-streams/de1f35e1-5370-4902-9fb5-88cfac856eae

OTLP endpoint -> https://api.demo-v2.galileocloud.io/otel/traces


## Step 2: Build a Pure OTel Pipeline

This is the heart of the customer pattern. Compare this cell to `init_telemetry(app)` in the customer's sample script — the shape is almost identical:

1. Create a `Resource` that identifies the service (picked up automatically by downstream analysis tools)
2. Build a `TracerProvider` and register it globally via `trace.set_tracer_provider(...)` — OpenInference's instrumentor will pick this up
3. Construct the OTLP exporter with the Galileo endpoint **and the three Galileo routing headers**
4. Wrap it in a `BatchSpanProcessor` (production default — async export)
5. Turn on `OpenAIInstrumentor` so any `openai.OpenAI().chat.completions.create(...)` call becomes an LLM span automatically

`uninstrument()` before `instrument()` makes this cell safe to re-run inside a notebook kernel without stacking multiple instrumentations on top of each other.

In [14]:
def configure_tracing():
    resource = Resource.create({
        'service.name': SERVICE_NAME,
        'service.version': '1.0.0',
        'deployment.environment': 'notebook',
    })

    tracer_provider = TracerProvider(resource=resource)

    # --- The ONLY Galileo-specific configuration below ---
    exporter = OTLPSpanExporter(
        endpoint=OTEL_TRACES_ENDPOINT,
        headers={
            'Galileo-API-Key': GALILEO_API_KEY,
            'project': PROJECT_NAME,
            'logstream': LOG_STREAM_NAME,
        },
    )
    # ------------------------------------------------------

    span_processor = BatchSpanProcessor(exporter)
    tracer_provider.add_span_processor(span_processor)
    trace.set_tracer_provider(tracer_provider)

    # OpenInference auto-instrumentation for OpenAI. The instrumentor finds
    # the global TracerProvider we just registered.
    instrumentor = OpenAIInstrumentor()
    try:
        instrumentor.uninstrument()
    except Exception:
        pass
    instrumentor.instrument(tracer_provider=tracer_provider)

    return tracer_provider, span_processor, exporter, instrumentor


def force_flush(timeout_millis: int = 40000) -> None:
    span_processor.force_flush(timeout_millis)


tracer_provider, span_processor, exporter, instrumentor = configure_tracing()
tracer = trace.get_tracer('scenario-11')
client = openai.OpenAI()

print('Pure OTel pipeline ready:')
print(f'  Service      -> {SERVICE_NAME}')
print(f'  Endpoint     -> {OTEL_TRACES_ENDPOINT}')
print(f'  Routing      -> project={PROJECT_NAME}, logstream={LOG_STREAM_NAME} (HTTP headers)')
print('  Exporter     -> OTLPSpanExporter (standard OTel)')
print('  Processor    -> BatchSpanProcessor (standard OTel)')
print('  Instrumentor -> OpenInference OpenAIInstrumentor')

Overriding of current TracerProvider is not allowed


Pure OTel pipeline ready:
  Service      -> galileo-pure-otel-demo
  Endpoint     -> https://api.demo-v2.galileocloud.io/otel/traces
  Routing      -> project=GalileoEval_S11_PureOTel, logstream=pure-otel (HTTP headers)
  Exporter     -> OTLPSpanExporter (standard OTel)
  Processor    -> BatchSpanProcessor (standard OTel)
  Instrumentor -> OpenInference OpenAIInstrumentor


## Step 3: Simplest Possible LLM Call

One call, auto-instrumented. The `OpenAIInstrumentor` produces an LLM span with the OpenInference `llm.*` attributes (model, prompt/completion tokens, input/output messages), which Galileo renders as a native LLM span in the console.

In [15]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant that explains technical concepts clearly.'},
        {'role': 'user', 'content': 'What is OpenTelemetry and why is it useful for AI applications?'},
    ],
    max_tokens=200,
)

force_flush()

print('Response:')
print(response.choices[0].message.content)
print(f'\nTokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out')
print(f'\n-> Trace flushed to Galileo. Check: {log_stream_url}')

Response:
OpenTelemetry is an open-source observability framework designed to help developers collect and manage telemetry data (such as metrics, traces, and logs) from applications. It provides a set of APIs, libraries, agents, and instrumentation to standardize the process of gathering performance and operational data from distributed systems.

### Key Features of OpenTelemetry:

1. **Unified Telemetry Collection**: OpenTelemetry combines traces, metrics, and logs into a single framework, which simplifies the integration and observability of applications.

2. **Vendor-Agnostic**: It is designed to work with multiple backends for data storage and analysis, meaning it can be integrated with a variety of observability platforms like Prometheus, Grafana, Jaeger, and more.

3. **Context Propagation**: OpenTelemetry supports distributed context propagation, enabling developers to track the flow of requests across microservices, which is crucial in AI applications that often use a microserv

## Step 4: Customer-Style Nested Spans — workflow → retriever → llm → tool

This cell builds one trace with four spans — three created by hand, the middle `llm` span contributed by the `OpenAIInstrumentor`.

**Galileo span classification relies on OTel GenAI semconv attributes:**

- **Retriever** must have `db.operation` = `"search"`. Output should use `gen_ai.output.messages` with a `{"documents": [...]}` content payload — Galileo extracts context/documents from retriever spans automatically.
- **Tool** must have `gen_ai.operation.name` = `"execute_tool"` and `gen_ai.tool.name`.
- **Workflow** uses `gen_ai.input.messages` / `gen_ai.output.messages` with user/assistant message format.

Token metrics are extracted from `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` on LLM spans (set automatically by the `OpenAIInstrumentor`).

The customer's business attributes (`workflow.name`, `usecase.id`, `step.name`, etc.) flow through as free-form OTel attributes, queryable in Galileo's log-stream filters.


In [16]:
user_question = 'What are the best practices for OpenTelemetry in LLM apps?'
workflow_name = 'research-agent'

with tracer.start_as_current_span(
    f'workflow:{workflow_name}',
    attributes={
        'gen_ai.system': 'custom-otel',
        'workflow.name': workflow_name,
        'usecase.id': 'demo-usecase-001',
        'sys.id': 'notebook-sys-id',
        'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
    },
) as workflow_span:

    # 1) Retriever span — `db.operation` = `search` is the primary discriminator.
    #    Output uses the `{"documents": [...]}` content shape in gen_ai.output.messages.
    retrieved_docs = [
        {'content': 'Always use a BatchSpanProcessor in production.', 'metadata': {'source': 'otel-docs', 'score': 0.91}},
        {'content': 'Set OTEL_SERVICE_NAME so spans carry a service identity.', 'metadata': {'source': 'otel-docs', 'score': 0.87}},
        {'content': 'Route spans with vendor headers instead of per-span attributes when possible.', 'metadata': {'source': 'galileo-docs', 'score': 0.84}},
    ]
    with tracer.start_as_current_span(
        'retriever:vector-search',
        attributes={
            'gen_ai.system': 'custom-otel',
            'db.operation': 'search',
            'http.url': 'https://vector-store.internal/query',
            'http.method': 'POST',
            'gen_ai.input.messages': json.dumps([{'role': 'user', 'content': user_question}]),
        },
    ) as retriever_span:
        retriever_span.set_attribute(
            'gen_ai.output.messages',
            json.dumps([{'role': 'assistant', 'content': {'documents': retrieved_docs}}]),
        )
        print(f'Retrieved {len(retrieved_docs)} documents')

    # 2) LLM span — created automatically by OpenAIInstrumentor, nested under workflow.
    context_text = '\n'.join(f'- {d["content"]}' for d in retrieved_docs)
    synthesis = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
            {'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {user_question}'},
        ],
        max_tokens=150,
    )
    answer = synthesis.choices[0].message.content

    # 3) Tool span — `gen_ai.operation.name` = `execute_tool` is the discriminator.
    step_name = 'format-final-answer'
    with tracer.start_as_current_span(
        f'tool:{step_name}',
        attributes={
            'gen_ai.system': 'custom-otel',
            'gen_ai.operation.name': 'execute_tool',
            'gen_ai.tool.name': step_name,
            'step.name': step_name,
            'step.operation': 'strip-whitespace',
            'gen_ai.tool.call.arguments': answer,
            'gen_ai.input.messages': json.dumps([{'role': 'tool', 'content': answer}]),
        },
    ) as tool_span:
        formatted = answer.strip()
        tool_span.set_attribute('gen_ai.tool.call.result', formatted)
        tool_span.set_attribute('gen_ai.output.messages', json.dumps([{'role': 'tool', 'content': formatted}]))

    workflow_span.set_attribute(
        'gen_ai.output.messages',
        json.dumps([{'role': 'assistant', 'content': formatted}]),
    )

force_flush()

print(f'\nFinal answer: {formatted}')
print('\n-> One trace exported with workflow -> retriever -> llm -> tool spans')
print(f'   Check: {log_stream_url}')

Retrieved 3 documents

Final answer: The best practices for OpenTelemetry in LLM apps include:

1. Always use a BatchSpanProcessor in production to optimize performance.
2. Set the OTEL_SERVICE_NAME environment variable to ensure spans have a service identity.
3. Route spans using vendor headers whenever possible, instead of relying on per-span attributes.

-> One trace exported with workflow -> retriever -> llm -> tool spans
   Check: https://console.demo-v2.galileocloud.io/project/32921366-fb69-4449-99d9-05453ebaebd9/log-streams/de1f35e1-5370-4902-9fb5-88cfac856eae


## Step 5: Reusable Decorators (remove the boilerplate)

The customer's sample has three nearly-identical `with tracer.start_as_current_span(...)` blocks. Once the pattern is clear, wrapping it in decorators makes application code dramatically cleaner — and ensures everyone on the team produces spans with the correct Galileo classification attributes.

Each decorator sets the **required discriminator** for its span type:

- `@otel_retriever` → `db.operation` = `"search"`, `gen_ai.output.messages` with `{"documents": [...]}`
- `@otel_tool` → `gen_ai.operation.name` = `"execute_tool"`, `gen_ai.tool.name`, `gen_ai.tool.call.*`
- `@otel_workflow` → `gen_ai.input/output.messages` in the GenAI message format


In [17]:
def _snapshot(args, kwargs):
    return json.dumps({'args': [str(a) for a in args], 'kwargs': {k: str(v) for k, v in kwargs.items()}})


def _msg(role, content):
    return json.dumps([{'role': role, 'content': content}])


def otel_workflow(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            input_str = _snapshot(args, kwargs)
            with tracer.start_as_current_span(
                f'workflow:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'workflow.name': name,
                    'gen_ai.input.messages': _msg('user', input_str),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                span.set_attribute('gen_ai.output.messages', _msg('assistant', str(result)))
                return result
        return wrapper
    return decorator


def otel_retriever(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            query = args[0] if args else kwargs.get('query', '')
            with tracer.start_as_current_span(
                f'retriever:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'db.operation': 'search',
                    'gen_ai.input.messages': _msg('user', str(query)),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                docs = result if isinstance(result, list) else [result]
                span.set_attribute(
                    'gen_ai.output.messages',
                    json.dumps([{'role': 'assistant', 'content': {'documents': docs}}]),
                )
                return result
        return wrapper
    return decorator


def otel_tool(name, **extra_attrs):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            input_str = _snapshot(args, kwargs)
            with tracer.start_as_current_span(
                f'tool:{name}',
                attributes={
                    'gen_ai.system': 'custom-otel',
                    'gen_ai.operation.name': 'execute_tool',
                    'gen_ai.tool.name': name,
                    'step.name': name,
                    'gen_ai.tool.call.arguments': input_str,
                    'gen_ai.input.messages': _msg('tool', input_str),
                    **extra_attrs,
                },
            ) as span:
                result = fn(*args, **kwargs)
                span.set_attribute('gen_ai.tool.call.result', str(result))
                span.set_attribute('gen_ai.output.messages', _msg('tool', str(result)))
                return result
        return wrapper
    return decorator


print('Decorators defined: @otel_workflow, @otel_retriever, @otel_tool')

Decorators defined: @otel_workflow, @otel_retriever, @otel_tool


## Step 6: Use the Decorators in a Pipeline

Now the agent code looks like normal Python. Tracing disappears from the logic and lives on the edges, exactly like in a well-instrumented production codebase.

In [18]:
@otel_retriever('search-knowledge-base', **{'http.url': 'https://kb.internal/search', 'http.method': 'POST'})
def search_knowledge_base(query):
    return [
        {'content': 'TracerProvider owns span processors and tracers.', 'metadata': {'source': 'otel-docs'}},
        {'content': 'Routing to Galileo uses the Galileo-API-Key, project, and logstream headers.', 'metadata': {'source': 'galileo-docs'}},
        {'content': 'OpenInference provides AI-specific span attributes on top of vanilla OTel.', 'metadata': {'source': 'openinference-docs'}},
    ]


@otel_tool('format-context', **{'step.operation': 'join-lines'})
def format_context(docs):
    return '\n'.join(f'- {doc["content"]}' for doc in docs)


@otel_tool('format-final-output', **{'step.operation': 'strip'})
def format_final_output(answer):
    return answer.strip()


@otel_workflow('knowledge-qa-pipeline', **{'usecase.id': 'qa-demo', 'sys.id': 'notebook-sys-id'})
def answer_question(query):
    docs = search_knowledge_base(query)
    context = format_context(docs)

    llm_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the context provided.'},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
        max_tokens=100,
    )

    return format_final_output(llm_response.choices[0].message.content)


final = answer_question('What is a TracerProvider?')
force_flush()

print(f'Answer: {final}')
print('\n-> Pipeline trace flushed to Galileo')
print(f'   Check: {log_stream_url}')

Answer: A TracerProvider is a component that manages the creation of tracers and span processors in distributed tracing systems. It is responsible for producing tracer instances used to create spans, which represent individual units of work in a trace.

-> Pipeline trace flushed to Galileo
   Check: https://console.demo-v2.galileocloud.io/project/32921366-fb69-4449-99d9-05453ebaebd9/log-streams/de1f35e1-5370-4902-9fb5-88cfac856eae


## Step 7: Flush and Verify

`BatchSpanProcessor` exports asynchronously. In a notebook an explicit `force_flush()` keeps the printed log-stream URL truthful — traces will be visible in the Galileo console before the next cell runs.

In [19]:
force_flush()

print('All spans flushed to Galileo')
print(f'View traces and metrics at: {log_stream_url}')

All spans flushed to Galileo
View traces and metrics at: https://console.demo-v2.galileocloud.io/project/32921366-fb69-4449-99d9-05453ebaebd9/log-streams/de1f35e1-5370-4902-9fb5-88cfac856eae


## Step 8: Clean Up

Clean shutdown matters as much as clean setup:

1. Uninstrument OpenAI so any later notebook cells (or a re-run of this notebook) start cleanly
2. Shut down the `TracerProvider` so the batch processor drains
3. Delete the Galileo project so the demo doesn't leave residue behind

In [10]:
instrumentor.uninstrument()
tracer_provider.shutdown()

delete_project(name=PROJECT_NAME)
print(f'Deleted project: {PROJECT_NAME}')

Deleted project: GalileoEval_S11_PureOTel
